### Silver to Gold — HR Domain (dim_broker)
**Author:** Virendra Tambavekar  
**Task:** Build Gold dimension table `dim_broker` from Silver  
**Domain:** HR (Broker)  
**Pipeline Stage:** Silver → Gold  
**Source:** `charles_schwab_retailbrokerage_dev_team_lemma.silver.broker` (50,000 rows)  
**Target:** `charles_schwab_retailbrokerage_dev_team_lemma.gold.dim_broker` (14,239 rows)  

**Transformations:**
- Filter: `JOB_CODE = 314` (brokers only)
- Surrogate Key: `SK_BrokerID = EMPLOYEE_ID`
- SCD-1 dimension: `IsCurrent = TRUE`, `EffectiveDate = batch_date`, `EndDate = 9999-12-31`
- No history tracking — always current state only

In [0]:
%run ../../02_common_utils/operations

In [0]:
from pyspark.sql import functions as F

In [0]:
# Configuration
CATALOG = "charles_schwab_retailbrokerage_dev_team_lemma"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"

SOURCE_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.broker"
TARGET_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.dim_broker"

print(f"Source: {SOURCE_TABLE}")
print(f"Target: {TARGET_TABLE}")

In [0]:
# Read from Silver
silver_df = spark.table(SOURCE_TABLE)
print(f"Silver (all employees) row count: {silver_df.count()}")
silver_df.printSchema()

In [0]:
# Extract carry-forwarded run_id from Silver layer
spark.sql(f"USE CATALOG {CATALOG}")
carried_run_id = str(silver_df.select("`_run_id`").first()[0])
print(f"Carry-forwarded run_id: {carried_run_id}")

# __ start_pipeline_run -- imported from operations
start_pipeline_run(spark=spark, run_id=carried_run_id, batch="ALL")

# __ log_domain_run_status -- imported from operations
log_domain_run_status(spark=spark, run_id=carried_run_id, batch="ALL", domain_name="HR", status="RUNNING")

# __ log_pipeline_message -- imported from operations
log_pipeline_message(spark=spark, run_id=carried_run_id, log_level="INFO", module="silver_to_gold_hr", message="Pipeline started: Silver to Gold transformation for HR domain (dim_broker)")

In [0]:
# Step 1: Filter to brokers only (JOB_CODE = 314)
brokers_df = silver_df.filter(F.col("JOB_CODE") == 314)
print(f"Brokers (JOB_CODE=314) row count: {brokers_df.count()}")

In [0]:
# Step 2: Build dim_broker Gold table
# SCD-1: IsCurrent = TRUE, EndDate = 9999-12-31
# SK_BrokerID = EMPLOYEE_ID (direct assignment)
# _run_id is carry-forwarded from the silver (upstream) layer

dim_broker_df = (
    brokers_df
    .select(
        F.col("EMPLOYEE_ID").alias("SK_BrokerID"),
        F.col("EMPLOYEE_ID").alias("BrokerID"),
        F.col("MANAGER_ID").alias("ManagerID"),
        F.col("FIRST_NAME").alias("FirstName"),
        F.col("LAST_NAME").alias("LastName"),
        F.col("MIDDLE_INITIAL").alias("MiddleInitial"),
        F.col("FULL_NAME").alias("FullName"),
        F.col("BRANCH_ID").alias("Branch"),
        F.col("OFFICE").alias("Office"),
        F.col("PHONE").alias("Phone"),
        # SCD-1 columns
        F.lit(True).alias("IsCurrent"),
        F.current_date().alias("EffectiveDate"),
        F.lit("9999-12-31").cast("date").alias("EndDate"),
        # Audit columns (carry-forwarded from upstream)
        F.col("_batch").alias("BatchID"),
        F.col("_run_id"),
        F.current_timestamp().alias("_load_ts")
    )
)

print("dim_broker schema:")
dim_broker_df.printSchema()
print(f"dim_broker row count: {dim_broker_df.count()}")

In [0]:
# Step 3: Write to Gold as CREATE OR REPLACE
dim_broker_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TARGET_TABLE)

print(f"Gold table written successfully: {TARGET_TABLE}")

In [0]:
# Step 4: Validation
source_count = brokers_df.count()
target_count = spark.table(TARGET_TABLE).count()

print(f"Source (Silver brokers): {source_count} rows")
print(f"Target (Gold dim_broker): {target_count} rows")
print(f"Status: {'MATCH' if target_count == source_count else f'MISMATCH (diff={target_count - source_count})'}")
print("\n--- Sample Data ---")
display(spark.table(TARGET_TABLE).limit(10))

In [0]:
# Step 5: Operations Logging

# __ log_pipeline_recon -- imported from operations
log_pipeline_recon(
    spark=spark,
    run_id=carried_run_id,
    batch_id="ALL",
    domain="HR",
    table_name="dim_broker",
    source_layer="silver",
    target_layer="gold",
    source_count=source_count,
    target_count=target_count
)

# __ log_audit_event -- imported from operations
log_audit_event(
    spark=spark,
    run_id=carried_run_id,
    batch="ALL",
    layer="gold",
    table_name="dim_broker",
    operation="OVERWRITE",
    rows_affected=target_count
)

# __ log_gold_recon -- imported from operations
log_gold_recon(
    spark=spark,
    run_id=carried_run_id,
    gold_table="dim_broker",
    expected_count=source_count,
    actual_count=target_count
)

# __ log_pipeline_message -- imported from operations
log_pipeline_message(spark=spark, run_id=carried_run_id, log_level="INFO", module="silver_to_gold_hr", message=f"Pipeline completed: {target_count} rows written to gold.dim_broker")

# __ log_domain_run_status -- imported from operations
log_domain_run_status(spark=spark, run_id=carried_run_id, batch="ALL", domain_name="HR", status="COMPLETED")

# __ end_pipeline_run -- imported from operations
end_pipeline_run(spark=spark, run_id=carried_run_id, status="SUCCESS")

print(f"Operations logging complete for run_id: {carried_run_id}")

In [0]:
def log_dq_result(spark: SparkSession, run_id: str, table_name: str, rule_name: str, failed_rows: int, total_rows: int):
    """
    Logs the outcome of a Data Quality (DQ) check.
    """
    status = 'PASS' if failed_rows == 0 else 'FAIL'
    
    spark.sql(f"""
        INSERT INTO operations.dq_results 
        (run_id, table_name, rule_name, failed_rows, total_rows, dq_status)
        VALUES ('{run_id}', '{table_name}', '{rule_name}', {failed_rows}, {total_rows}, '{status}')
    """)